In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses

In [ ]:
MAX_VOCAB=1000
CONTEXT_WIN=20
EMBED_DIM=64
HEADS=2
FEED_FORWARD=64
TRANSFORMER_BLOCKS=2
BATCH_SIZE=16
EPOCH=150

train_text = [
    'i hate java',
    'python is fire',
    'AI is the future',
    'TUS Students are cooked',
    'Add me on linkedIn',
    'MongoDB kinda lame',
    'coding is fire',
    'I love coding',
    'Should I submit it on Kaggle?',
    'AI is fire',
    'Vibecoders are not true coders',
    'Python is AI standart',
    'I dont know what to add',
    'Stranger things is fire',
    'Math is cool',
    'hope there will be no Java in Ericcson',
    'machine learning is fascinating',
    'deep learning requires a lot of data',
    'neural networks are powerful',
    'I need to debug my code',
    'stack overflow is my best friend',
    'git commit and push',
    'data science is a great career',
    'I enjoy building models',
    'tensorflow makes AI easier',
    'pytorch is also very popular',
    'natural language processing is amazing',
    'transformers changed the world of AI',
    'I want to build a large language model',
    'coding late at night is a vibe'
]

tokenizer = layers.TextVectorization(
    max_tokens = MAX_VOCAB,
    output_mode = 'int',
    output_sequence_length =CONTEXT_WIN+1
)

tokenizer.adapt(train_text)
vocab = tokenizer.get_vocabulary()

def prep_data(train_text):
  seq = tokenizer(train_text)
  X_target = seq[:,:-1]
  y_label = seq[:,1:]

  return X_target, y_label

X_train, y_train = prep_data(train_text)


ValueError: Unknown value for `output_mode` argument of TextVectorization. Allowed values are: ('int', 'one_hot', 'multi_hot', 'count', 'tf_idf'). Received: output_mode=<class 'int'>

In [ ]:
def perplexity(true, pred):
  return tf.exp(tf.reduce_mean(losses.sparse_categorical_crossentropy(true, pred, from_logits=True)))

class TokenPositionEmbedding(layers.Layer):
  def __init__(self, context_win, max_vocab, embed_dim, **kwargs) -> None:
    super().__init__(**kwargs)
    self.token_embed = layers.Embedding(input_dim=max_vocab, output_dim=embed_dim)
    self.position_embed = layers.Embedding(input_dim=context_win, output_dim=embed_dim)

  def call(self, x):
    context_win = tf.shape(x)[-1]
    positions = self.position_embed(tf.range(start=0, limit=context_win, delta=1))

    return self.token_embed(x) + positions


NameError: name 'layers' is not defined

In [ ]:
class TransformerBlock(layers.Layer):
  def __init__(self, embed_dim, heads, feed_forward, rate=0.1, **kwargs) -> None:
    super().__init__(**kwargs)
    self.attention = layers.MultiHeadAttention(num_heads=heads, key_dim=embed_dim)
    self.feed_forward_net = models.Sequential([
        layers.Dense(feed_forward, activation='relu'),
        layers.Dense(embed_dim)
    ])

    self.norm1 = layers.LayerNormalization(epsilon=1e-6)
    self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    self.drop1 = layers.Dropout(rate)
    self.drop2 = layers.Dropout(rate)

  def call(self, inputs, training=False):
    attention_output = self.attention(inputs, inputs, use_causal_mask=True)
    attention_output = self.drop1(attention_output, training=training)
    output = self.norm1(inputs + attention_output)

    feed_forward_out = self.feed_forward_net(output)
    feed_forward_out = self.drop2(feed_forward_out, training=training)

    return self.norm2(feed_forward_out + output)


NameError: name 'layers' is not defined

In [ ]:
class LLM(models.Model):
  def __init__(self, context_win, max_vocab, embed_dim, heads, feed_forward, num_transformer_blocks, batch_size, epoch, **kwargs):
    super().__init__(**kwargs)
    self.embed_layer = TokenPositionEmbedding(context_win, max_vocab, embed_dim)
    self.blocks = [
        TransformerBlock(embed_dim, heads, feed_forward) for _ in range(num_transformer_blocks)
    ]
    self.dense_out = layers.Dense(max_vocab)

  def call(self, inputs, training=False):
    x = self.embed_layer(inputs)
    for block in self.blocks:
      x = block(x, training=training)
    return self.dense_out(x)

def gen(model, prompt, length=10, temperature=1.0, top_k=5):
  input_tensor = tokenizer([prompt])
  tokens = [token for token in input_tensor.numpy()[0] if token != 0]

  gen_text = prompt

  for _ in range(length):
    context_tok = tokens[-CONTEXT_WIN:]
    input_data = tf.convert_to_tensor([context_tok])

    preds = model(input_data, training=False)
    next_logits = preds[0, -1, :]
    next_logits = next_logits / (temperature + 1e-7)
    top_vals, top_ind = tf.math.top_k(next_logits, k=top_k)

    top_probs = tf.nn.softmax(top_vals).numpy()

    next_ind = np.random.choice(top_ind.numpy(), p=top_probs)
    if next_ind == 0 and len(tokens) > 0 and len(top_ind.numpy()) > 1:
      next_ind = top_ind.numpy()[1]

    tokens.append(next_ind)
    gen_text += ' ' + vocab[next_ind]

  return gen_text


NameError: name 'models' is not defined

In [ ]:
def main():
  test_model = LLM(
      CONTEXT_WIN,
      len(vocab),
      EMBED_DIM,
      HEADS,
      FEED_FORWARD,
      TRANSFORMER_BLOCKS,
      BATCH_SIZE,
      EPOCH
  )

  test_model.compile(
      optimizer='adam',
      loss = losses.SparseCategoricalCrossentropy(from_logits = True),
      metrics = [perplexity]
  )

  test_model.fit(
      x=X_train, y=y_train,
      batch_size = BATCH_SIZE,
      epochs = EPOCH,
      verbose=1
  )

  test_input1 = input('\nEnter first text to test: ')
  test_input2 = input('Enter second text to test: ')

  print(f'\nPrompt : {test_input1}\nGenerated text: {gen(test_model, test_input1, length=15)}')
  print(f'\nPrompt : {test_input2}\nGenerated text: {gen(test_model, test_input2, length=15)}')

main()
